# Task6 机器学习选股策略与季度回测

本 Notebook 展示第 6 次课作业流程：加载上一课模型样本，衍生未来季度收益率，重新定义应变量，训练四种模型，并把预测分数转换为 Top 20 季度选股策略。

In [ ]:
from pathlib import Path
import pandas as pd

ROOT = Path.cwd().parents[0] if Path.cwd().name == 'notebooks' else Path.cwd()
TASK6 = ROOT / 'task6'
DATA = TASK6 / 'data'
OUTPUTS = TASK6 / 'outputs'

## 1. 加载样本与收益率数据

样本来自上一课，收益率由样本日下一交易日开盘价买入、下一季度末收盘价卖出计算。

In [ ]:
sample = pd.read_csv(DATA / 'model_data_stock.csv')
returns = pd.read_csv(DATA / 'price_returns.csv')
dataset = pd.read_csv(OUTPUTS / 'model_dataset_with_returns.csv')
sample.shape, returns.shape, dataset.shape

## 2. 应变量定义

`future_return = sell_close / buy_open - 1`，`target_up = 1 if future_return > 0 else 0`。核心回测使用扣除交易成本后的 `net_return`。

In [ ]:
dataset[['Date','Code','buy_date','sell_date','gross_return','net_return','future_return','target_up']].head()

## 3. 增长综合分

增长综合分使用净利润同比增长率 40%，营业利润、营业总收入、基本每股收益同比增长率各 20%。

In [ ]:
growth_cols = ['净利润同比增长率','营业利润(同比增长率)','营业总收入(同比增长率)','基本每股收益(同比增长率)','growth_score']
dataset[growth_cols].describe().T

## 4. 模型评估

In [ ]:
metrics = pd.read_csv(OUTPUTS / 'model_metrics.csv')
metrics

## 5. Top 20 策略回测

每个测试季度按模型分数动态选择前 20 只股票，组合等权，收益率扣除买入成本率 0.0005441 和卖出成本率 0.0010441。

In [ ]:
quarterly = pd.read_csv(OUTPUTS / 'quarterly_returns.csv')
backtest = pd.read_csv(OUTPUTS / 'backtest_metrics.csv')
quarterly, backtest

## 6. 持仓明细与模型对比

In [ ]:
holdings = pd.read_csv(OUTPUTS / 'selected_holdings.csv')
holdings[['model_label','Date','rank','ts_code','score','gross_return','net_return']].head(30)